# Plan-and-Execute Agent

## Review

The **ReAct** agent (Notebook 5) interleaves reasoning and acting step-by-step.

## Goals

The **Plan-and-Execute** pattern separates planning from execution:

1. **Plan** — generate a full plan upfront (list of steps)
2. **Execute** — carry out each step sequentially
3. **Replan** — after each step, optionally revise the remaining plan

This is more powerful than ReAct for complex multi-step tasks because:
- The LLM sees the full picture before acting
- Steps can be validated before execution
- Failed steps trigger replanning, not just retry

```
User Request → Planner → [Step 1] → [Step 2] → ... → Final Answer
                  ↑                                        |
                  └──── Replanner (if needed) ─────────────┘
```

In [ ]:
%run ../langchain_common.py

## Define the State

Our state tracks:
- The original request
- The current plan (list of steps)
- Results from completed steps
- The final response

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from typing_extensions import TypedDict
from typing import List, Optional
from IPython.display import Image, display
import json

In [ ]:
class PlanExecuteState(TypedDict):
    """State for the plan-and-execute agent."""
    request: str                    # Original user request
    plan: List[str]                 # List of planned steps
    completed_steps: List[str]      # Results of completed steps
    current_step_index: int         # Which step we're on
    response: str                   # Final response to user

## Define Tools

The executor will have access to these tools for carrying out plan steps.

In [ ]:
@tool
def search(query: str) -> str:
    """Search for information on a topic."""
    knowledge = {
        "population of france": "France has a population of approximately 68 million.",
        "capital of france": "The capital of France is Paris.",
        "area of france": "France covers approximately 643,801 square kilometers.",
        "gdp of france": "France's GDP is approximately $2.78 trillion (2023).",
        "population of germany": "Germany has a population of approximately 84 million.",
        "capital of germany": "The capital of Germany is Berlin.",
    }
    for key, value in knowledge.items():
        if key in query.lower():
            return value
    return f"Found general info about: {query}"

@tool
def calculate(expression: str) -> str:
    """Calculate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

print("Tools defined: search, calculate")

## The Planner Node

The planner receives the user's request and generates a step-by-step plan.

In [ ]:
def planner(state: PlanExecuteState) -> dict:
    """Generate a plan from the user's request."""
    planner_prompt = SystemMessage(content="""You are a planner. Given a user request, break it down into simple, sequential steps.

Rules:
- Each step should be a single, atomic action
- Steps should be in logical order
- Use 2-5 steps (no more)
- Available actions: search for info, calculate math, summarize/compare results

Respond with ONLY a JSON list of step strings. Example:
["Search for the population of France", "Search for the population of Germany", "Compare the two populations"]""")
    
    response = llm_noreason.invoke([
        planner_prompt,
        HumanMessage(content=state["request"])
    ])
    
    # Parse the plan
    try:
        plan = json.loads(response.content)
    except json.JSONDecodeError:
        # Fallback: treat entire response as single step
        plan = [response.content]
    
    print(f"\U0001f4cb Plan created with {len(plan)} steps:")
    for i, step in enumerate(plan, 1):
        print(f"   {i}. {step}")
    
    return {"plan": plan, "completed_steps": [], "current_step_index": 0}

## The Executor Node

The executor takes the current step and carries it out using available tools.

In [ ]:
executor_tools = [search, calculate]
executor_llm = llm_noreason.bind_tools(executor_tools)

def executor(state: PlanExecuteState) -> dict:
    """Execute the current step of the plan."""
    idx = state["current_step_index"]
    current_step = state["plan"][idx]
    
    # Build context from previous steps
    context = ""
    if state["completed_steps"]:
        context = "Previous results:\n" + "\n".join(state["completed_steps"]) + "\n\n"
    
    exec_prompt = SystemMessage(content=f"""Execute this step using available tools. Be concise.
{context}Current step: {current_step}""")
    
    response = executor_llm.invoke([exec_prompt, HumanMessage(content=current_step)])
    
    # Handle tool calls
    if response.tool_calls:
        from langchain_core.messages import ToolMessage
        results = []
        for tc in response.tool_calls:
            tool_fn = search if tc["name"] == "search" else calculate
            result = tool_fn.invoke(tc["args"])
            results.append(result)
        step_result = f"Step {idx+1}: {current_step} \u2192 {'; '.join(results)}"
    else:
        step_result = f"Step {idx+1}: {current_step} \u2192 {response.content}"
    
    print(f"  \u2705 {step_result}")
    
    completed = state["completed_steps"] + [step_result]
    return {"completed_steps": completed, "current_step_index": idx + 1}

## The Replanner / Finalizer

After each step, we check: are we done? If so, generate the final answer. If not, continue.

In [ ]:
def should_continue(state: PlanExecuteState) -> str:
    """Check if all steps are complete."""
    if state["current_step_index"] >= len(state["plan"]):
        return "finalizer"
    return "executor"

def finalizer(state: PlanExecuteState) -> dict:
    """Generate final response from all completed steps."""
    final_prompt = SystemMessage(content="""Based on the completed steps below, provide a concise final answer to the user's original request.

Completed steps:
""" + "\n".join(state["completed_steps"]))
    
    response = llm_noreason.invoke([
        final_prompt,
        HumanMessage(content=state["request"])
    ])
    
    print(f"\n\U0001f4dd Final Answer: {response.content}")
    return {"response": response.content}

## Build the Plan-and-Execute Graph

In [ ]:
# Build the graph
builder = StateGraph(PlanExecuteState)

# Add nodes
builder.add_node("planner", planner)
builder.add_node("executor", executor)
builder.add_node("finalizer", finalizer)

# Add edges
builder.add_edge(START, "planner")
builder.add_edge("planner", "executor")
builder.add_conditional_edges("executor", should_continue, {"executor": "executor", "finalizer": "finalizer"})
builder.add_edge("finalizer", END)

# Compile
plan_execute_graph = builder.compile()
display(Image(plan_execute_graph.get_graph().draw_mermaid_png()))

## Test: Multi-Step Research Task

In [ ]:
result = plan_execute_graph.invoke({
    "request": "Compare the populations of France and Germany. Which is larger and by how much?"
})

print("\n" + "="*50)
print("FINAL RESPONSE:")
print("="*50)
print(result["response"])

## ReAct vs Plan-and-Execute

| Aspect | ReAct | Plan-and-Execute |
|--------|-------|------------------|
| **Planning** | Implicit (one step at a time) | Explicit (full plan upfront) |
| **Visibility** | Can't see ahead | Full plan visible |
| **Error Recovery** | Retry same step | Replan remaining steps |
| **Token Efficiency** | May wander | Focused execution |
| **Best For** | Simple tasks, exploration | Complex multi-step tasks |
| **Human Review** | Hard to preview | Can review plan before execution |

### When to Use Plan-and-Execute

- Tasks requiring 3+ coordinated steps
- When you want human review of the plan before execution
- When errors in early steps should change later steps
- When you need an audit trail of the reasoning process